# NLP

NLP, ou Processamento de Linguagem Natural, e uma area da ciencia de dados que estuda como transformar textos em informacao estruturada. Com essas tecnicas, conseguimos analisar noticias, identificar temas, contar palavras, comparar documentos e preparar dados textuais para modelos de aprendizado de maquina.

In [11]:
df = pd.read_csv("C:/Users/isabe/Perspectiva_dados/PLN/df_quotes.csv")
print(len(df))
print(df)

100
                                                frase               autor  \
0   “The world as we have created it is a process ...     Albert Einstein   
1   “It is our choices, Harry, that show what we t...        J.K. Rowling   
2   “There are only two ways to live your life. On...     Albert Einstein   
3   “The person, be it gentleman or lady, who has ...         Jane Austen   
4   “Imperfection is beauty, madness is genius and...      Marilyn Monroe   
..                                                ...                 ...   
95  “You never really understand a person until yo...          Harper Lee   
96  “You have to write the book that wants to be w...   Madeleine L'Engle   
97  “Never tell the truth to people who are not wo...          Mark Twain   
98        “A person's a person, no matter how small.”           Dr. Seuss   
99  “... a mind needs books as a sword needs a whe...  George R.R. Martin   

                                                 tags  
0    ['change',

In [1]:
import json
from pathlib import Path

import pandas as pd

noticias = []

for arquivo in sorted(Path("data").glob("*.json")):
    with arquivo.open(encoding="utf-8") as f:
        noticias.append(json.load(f))

df = pd.DataFrame(noticias)

print(f"Antes: {len(df)} noticias")

df = df.drop_duplicates(subset="url").reset_index(drop=True)

print(f"Depois: {len(df)} noticias")

df.head()

Antes: 0 noticias
Depois: 0 noticias


""


## Trabalhando apenas com o texto

Por enquanto, vamos trabalhar somente com a coluna `texto`, que contem o corpo completo de cada noticia. As outras colunas, como `titulo`, `descricao`, `tags` e `data`, continuam no `DataFrame`, mas vamos deixar elas de lado neste primeiro momento para entender melhor o conteudo textual.

## Passos da analise

Vamos preparar os textos aos poucos:

1. Limpar os textos.
2. Remover palavras muito comuns.
3. Criar uma representacao Bag of Words.
4. Contar palavras frequentes.
5. Transformar textos em numeros para analises posteriores.

## 1. Limpeza basica

Na celula abaixo:

- `wordpunct_tokenize` separa o texto em palavras e pontuacao.
- `texto.lower()` coloca tudo em minusculas.
- `unidecode(texto)` troca letras acentuadas por letras sem acento.
- `token.isalnum()` mantem apenas letras e numeros.
- `" ".join(tokens)` junta os tokens em uma frase limpa.
- `.apply(limpar_texto)` aplica a funcao em todas as noticias.

Exemplo: `"Olá, Senado!"` vira `"ola senado"`.

In [12]:
from nltk.tokenize import wordpunct_tokenize
from unidecode import unidecode


def limpar_texto(texto):
    texto = texto.lower()
    texto = unidecode(texto)
    tokens = wordpunct_tokenize(texto)
    tokens = [token for token in tokens if token.isalnum()]
    return " ".join(tokens)


df["frase_limpa"] = df["frase"].apply(limpar_texto)

df[["frase", "frase_limpa"]].head()

,frase,frase_limpa
0,“The world as we have created it is a process ...,the world as we have created it is a process o...
1,"“It is our choices, Harry, that show what we t...",it is our choices harry that show what we trul...
2,“There are only two ways to live your life. On...,there are only two ways to live your life one ...
3,"“The person, be it gentleman or lady, who has ...",the person be it gentleman or lady who has not...
4,"“Imperfection is beauty, madness is genius and...",imperfection is beauty madness is genius and i...


## 2. Removendo stopwords

Stopwords sao palavras muito comuns, como `a`, `o`, `de`, `para` e `que`. Elas aparecem muito, mas geralmente ajudam pouco a entender o tema de um texto.

Na celula abaixo:

- `stopwords.words("portuguese")` carrega stopwords em portugues.
- `texto.split()` separa o texto limpo em palavras.
- `token not in stopwords_pt` remove as palavras muito comuns.
- `.str.join(" ")` junta os tokens restantes em um texto sem stopwords.
- `.apply(remover_stopwords)` aplica a funcao em todas as noticias.

Exemplo: `"o senador falou com a imprensa"` vira `["senador", "falou", "imprensa"]`.

In [13]:
import nltk

from nltk.corpus import stopwords


nltk.download("stopwords", quiet=True)

stopwords_pt = stopwords.words("english")
stopwords_pt = [unidecode(palavra) for palavra in stopwords_pt]
stopwords_pt = set(stopwords_pt)


def remover_stopwords(texto):
    tokens = texto.split()
    tokens = [token for token in tokens if token not in stopwords_pt]
    return tokens


df["tokens_sem_stopwords"] = df["frase_limpa"].apply(remover_stopwords)
df["frase_sem_stopwords"] = df["tokens_sem_stopwords"].str.join(" ")

df[["frase_limpa", "tokens_sem_stopwords", "frase_sem_stopwords"]].head()

,frase_limpa,tokens_sem_stopwords,frase_sem_stopwords
0,the world as we have created it is a process o...,"[world, created, process, thinking, cannot, ch...",world created process thinking cannot changed ...
1,it is our choices harry that show what we trul...,"[choices, harry, show, truly, far, abilities]",choices harry show truly far abilities
2,there are only two ways to live your life one ...,"[two, ways, live, life, one, though, nothing, ...",two ways live life one though nothing miracle ...
3,the person be it gentleman or lady who has not...,"[person, gentleman, lady, pleasure, good, nove...",person gentleman lady pleasure good novel must...
4,imperfection is beauty madness is genius and i...,"[imperfection, beauty, madness, genius, better...",imperfection beauty madness genius better abso...


## 3. Bag of Words

No Bag of Words, cada linha representa uma noticia e cada coluna representa uma palavra. O valor indica quantas vezes aquela palavra apareceu na noticia.

Exemplo: `"senador falou senador"` teria `senador = 2` e `falou = 1`.

In [14]:
from sklearn.feature_extraction.text import CountVectorizer
vectorizer = CountVectorizer()
matriz_bow = vectorizer.fit_transform(df["frase_sem_stopwords"])

df_bow = pd.DataFrame(
    matriz_bow.toarray(),
    columns=vectorizer.get_feature_names_out()
)

df_bow.head()

ModuleNotFoundError: No module named 'sklearn'

### Removendo colunas com numeros

Agora vamos remover qualquer coluna cujo nome tenha pelo menos um numero.

In [26]:
colunas_com_numeros = [col for col in df_bow.columns if any(char.isdigit() for char in col)]

df_bow = df_bow.drop(columns=colunas_com_numeros)

print(f"{len(colunas_com_numeros)} colunas removidas")
df_bow.head()

46 colunas removidas


,abertura,abordadas,abraao,abril,abrir,absolvicao,abuso,ac,acabou,acao,...,votarmos,votos,voz,vulnerabilidades,weverton,wilder,wolney,xingamentos,zequinha,zettel
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,1,0,0,0,0,1,0,0,...,1,1,2,0,5,0,0,1,1,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0
3,0,0,0,0,0,0,0,0,0,2,...,0,1,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,2,...,0,0,0,0,0,0,0,0,0,4


### Criando o DataFrame final

Agora vamos juntar os metadados das noticias com as colunas do Bag of Words. Para evitar conflito de nomes, as colunas do Bag of Words recebem o prefixo `bow_`.

In [27]:
metadados = df[["data", "tags", "titulo", "url"]].reset_index(drop=True)
bow_com_prefixo = df_bow.add_prefix("bow_").reset_index(drop=True)

df_final = pd.concat([metadados, bow_com_prefixo], axis=1)

df_final.head()

,data,tags,titulo,url,bow_abertura,bow_abordadas,bow_abraao,bow_abril,bow_abrir,bow_absolvicao,...,bow_votarmos,bow_votos,bow_voz,bow_vulnerabilidades,bow_weverton,bow_wilder,bow_wolney,bow_xingamentos,bow_zequinha,bow_zettel
0,2026-03-30T17:31:28-03:00,"[Agricultura familiar , Emendas parlamentares,...",Chico Rodrigues destaca ações em Roraima e cri...,https://www12.senado.leg.br/noticias/materias/...,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,2026-03-27T19:39:12-03:00,"[Aposentados, INSS, Pensionistas]","Com mais de 4 mil páginas, relatório da CPMI p...",https://www12.senado.leg.br/noticias/materias/...,0,0,1,0,0,0,...,1,1,2,0,5,0,0,1,1,0
2,2026-03-24T16:59:17-03:00,"[Alexandre de Moraes, Jair Bolsonaro, Procurad...",Wilder defende prisão domiciliar para Jair Bol...,https://www12.senado.leg.br/noticias/materias/...,0,0,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0
3,2026-03-26T21:14:18-03:00,"[André Mendonça, Aposentados, Corrupção, INSS,...",CPMI do INSS: relatório será lido e pode ser v...,https://www12.senado.leg.br/noticias/materias/...,0,0,0,0,0,0,...,0,1,0,0,0,0,0,0,0,0
4,2026-03-26T20:05:27-03:00,"[Aposentados, Conselho de Controle de Atividad...",CPMI do INSS aprova mais uma convocação e queb...,https://www12.senado.leg.br/noticias/materias/...,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,4


## Exemplo de analise

Agora podemos fazer uma analise simples das palavras do Bag of Words: quais aparecem mais, quais aparecem menos e quantas palavras diferentes temos no vocabulario.

In [28]:
colunas_bow = [coluna for coluna in df_final.columns if coluna.startswith("bow_")]

frequencia_palavras = df_final[colunas_bow].sum().sort_values(ascending=False)

print(f"Total de palavras diferentes: {len(frequencia_palavras)}")

frequencia_palavras.head(10)

Total de palavras diferentes: 1681


bow_cpmi           56
bow_senado         52
bow_senador        45
bow_presidente     42
bow_ex             37
bow_ministro       34
bow_agencia        32
bow_inss           31
bow_prorrogacao    31
bow_stf            30
dtype: int64

In [29]:
frequencia_palavras.tail(10)

bow_adequada          1
bow_adentrar          1
bow_acusado           1
bow_acusacao          1
bow_acrescentando     1
bow_acrescenta        1
bow_aconteceu         1
bow_acontecendo       1
bow_acompanhou        1
bow_acompanhamento    1
dtype: int64

In [30]:
documentos_por_palavra = (df_final[colunas_bow] > 0).sum().sort_values(ascending=False)
documentos_por_palavra.index = documentos_por_palavra.index.str.replace("bow_", "", regex=False)

documentos_por_palavra.head(10)

agencia       16
feira         16
reproducao    16
autorizada    16
mediante      16
senado        16
senador       16
citacao       16
nesta         13
presidente    12
dtype: int64

In [32]:
df_final["palavras_unicas"] = (df_final[colunas_bow] > 0).sum(axis=1)

df_final[["titulo", "palavras_unicas"]].sort_values("palavras_unicas", ascending=True).head(10)

,titulo,palavras_unicas
9,Veneziano celebra 60 anos de fundação do MDB,80
8,Cancelada reunião da CPMI do INSS desta quarta...,87
7,Cleitinho defende revogação da condenação de B...,96
12,Girão diz ser vítima de censura e critica Comu...,97
5,Girão pede ao STF a abertura da CPI do Banco M...,118
15,CPI do Crime repudia anulação da quebra de sig...,126
2,Wilder defende prisão domiciliar para Jair Bol...,126
13,Cancelada reunião da CPMI do INSS desta segunda,140
10,Kajuru manifesta apoio à prisão domiciliar par...,154
0,Chico Rodrigues destaca ações em Roraima e cri...,160
